In [15]:
#from pyspark.sql.functions import col, sum
#from pyspark.sql.functions import to_timestamp
#from pyspark.sql.functions import when

### !!! RUN THIS CELL before all other cells !!! ###

from pyspark.sql import functions as F
from pyspark.sql.window import Window
import math
 
# Deduplicator helper: keep first occurrence of a key column
def dedup(df, key_col: str):
    w = Window.partitionBy(key_col).orderBy(F.monotonically_increasing_id())
    return (df.withColumn("_rn", F.row_number().over(w))
              .filter(F.col("_rn") == 1)
              .drop("_rn"))
 
# Deduplicator helper: composite key (two columns) 
def dedup2(df, col1: str, col2: str):
    w = Window.partitionBy(col1, col2).orderBy(F.monotonically_increasing_id())
    return (df.withColumn("_rn", F.row_number().over(w))
              .filter(F.col("_rn") == 1)
              .drop("_rn"))
 
# Write to silver managed Delta table
def write_silver(df, table_name: str):
    full_name = f"silver_lakehouse.dbo.silver_{table_name}"
    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(full_name))
    count = spark.read.table(full_name).count()
    print(f"✅  {full_name}: {count:,} rows written")
 
# Null count checker
def null_report(df, label=""):
    print(f"\n Null counts — {label}")
    df.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ]).show(truncate=False)
 
print("✅  Helpers loaded. Ready to run cleaning cells.")
 

StatementMeta(, 8f8003aa-7acd-4e3c-8ee0-165d97e9724a, 17, Finished, Available, Finished, False)

✅  Helpers loaded. Ready to run cleaning cells.


In [16]:
## 1.0 Cleaning Orders Table

orders_df = spark.read.table("bronze_orders")
print(f"bronze_orders: {orders_df.count():,} rows")


#1.1 Removing rows with missing essential IDs
orders_df = orders_df.filter(
    F.col("order_id").isNotNull() & F.col("customer_id").isNotNull()
)


#1.2 Trim to remove spaces at start/end of cells & nulify empty order_status
orders_df = orders_df.withColumn("order_status", F.trim(F.col("order_status")))
orders_df = orders_df.withColumn(
    "order_status",
    F.when(F.col("order_status") == "", None).otherwise(F.col("order_status"))
)


#1.3 Remove duplicate order_id rows (only keep 1st occurence)
orders_df = dedup(orders_df, "order_id")

#1.4 Insert filtered data into silver table
write_silver(orders_df, "orders")           

StatementMeta(, 8f8003aa-7acd-4e3c-8ee0-165d97e9724a, 18, Finished, Available, Finished, False)

bronze_orders: 99,441 rows
✅  silver_lakehouse.dbo.silver_orders: 99,441 rows written


In [17]:
## 2.0 Clean Order_reviews Table

order_reviews_df = spark.read.table("bronze_order_reviews")
print(f"bronze_order_reviews: {order_reviews_df.count():,} rows")


# 2.1 Trim whitespace & convert empty strings -> NULL
for c in ("review_comment_title", "review_comment_message"):
    order_reviews_df = order_reviews_df.withColumn(c, F.trim(F.col(c)))
    order_reviews_df = order_reviews_df.withColumn(
        c,
        F.when(F.col(c) == "", None).otherwise(F.col(c))
    )


# 2.2 Remove rows with missing essential IDs
order_reviews_df = order_reviews_df.filter(
    F.col("review_id").isNotNull() & F.col("order_id").isNotNull()
)


# 2.3 Remove duplicate review_id rows, only keep 1st occurence
order_review_df = dedup(order_reviews_df, "review_id")

write_silver(order_review_df, "order_reviews")

StatementMeta(, 8f8003aa-7acd-4e3c-8ee0-165d97e9724a, 19, Finished, Available, Finished, False)

bronze_order_reviews: 104,162 rows
✅  silver_lakehouse.dbo.silver_order_reviews: 101,925 rows written


In [18]:
## 3.0 Clean Order_items Table

order_items_df = spark.read.table("bronze_order_items")
initial_count = order_items_df.count()
print(f"bronze_order_items: {initial_count:,} rows")
 
# 3.1 Check duplicates for order_id, product_id 
print("\nDuplicate (order_id, product_id) pairs before clean:")
(order_items_df
 .groupBy("order_id", "product_id")
 .agg(F.count("*").alias("row_count"))
 .filter(F.col("row_count") > 1)
 .orderBy(F.col("row_count").desc())
 .show(10))
 
# 3.2 Remove duplicate using dedup2 (for 2 columns), only keep 1st occurrence
order_items_df = dedup2(order_items_df, "order_id", "product_id")
 
final_count = order_items_df.count()
print(f"\nRows removed: {initial_count - final_count:,}")
 
# #3.3 Final check if cleaning was successful
remaining = (order_items_df
             .groupBy("order_id", "product_id")
             .agg(F.count("*").alias("row_count"))
             .filter(F.col("row_count") > 1)
             .count())
print(f"Remaining duplicates (expect 0): {remaining}")
 
write_silver(order_items_df, "order_items")

StatementMeta(, 8f8003aa-7acd-4e3c-8ee0-165d97e9724a, 20, Finished, Available, Finished, False)

bronze_order_items: 112,650 rows

Duplicate (order_id, product_id) pairs before clean:
+--------------------+--------------------+---------+
|            order_id|          product_id|row_count|
+--------------------+--------------------+---------+
|1b15974a0141d54e3...|ee3d532c8a4386797...|       20|
|ab14fdcfbe524636d...|9571759451b1d780e...|       20|
|428a2f660dc84138d...|89b190a046022486c...|       15|
|9ef13efd6949e4573...|37eb69aca8718e843...|       15|
|9bdc4d4c71aa1de46...|44a5d24dd383324a4...|       14|
|73c8ab38f07dc9438...|422879e10f4668299...|       14|
|37ee401157a3a0b28...|d34c07a2d817ac73f...|       13|
|2c2a19b5703863c90...|03e1c946c0ddfc587...|       12|
|3a213fcdfe7d98be7...|a62e25e09e05e6faf...|       12|
|6c355e2913545fa6f...|32e18e89237933ebd...|       11|
+--------------------+--------------------+---------+
only showing top 10 rows


Rows removed: 10,225
Remaining duplicates (expect 0): 0
✅  silver_lakehouse.dbo.silver_order_items: 102,425 rows written


In [19]:
## 4.0 Clean Products & product_category_name_translation


products_df = spark.read.table("bronze_products")
trans_df    = spark.read.table("bronze_product_category_name_translation")
 
print(f"bronze_products:     {products_df.count():,} rows")
print(f"bronze_translation:  {trans_df.count():,} rows")
 
# --- Products ---
# 4.1.1 Trim
products_df = products_df.withColumn(
    "product_category_name", F.trim(F.col("product_category_name"))
)

# 4.1.2 Nullify empty and literal 'unknown'
products_df = products_df.withColumn(
    "product_category_name",
    F.when(F.col("product_category_name").isin("", "unknown"), None)
     .otherwise(F.col("product_category_name"))
)

# 4.1.3 Assign 'unknown' to NULLs (after nullifying so we get a clean slate)
products_df = products_df.withColumn(
    "product_category_name",
    F.when(F.col("product_category_name").isNull(), "unknown")
     .otherwise(F.col("product_category_name"))
)

# 4.1.4 Remove missing product_id
products_df = products_df.filter(F.col("product_id").isNotNull())

# 4.1.5 Deduplicate on product_id to remove 
products_df = dedup(products_df, "product_id")
 
write_silver(products_df, "products")
 


# --- Product name translation table ---
# 4.2.1 Double check if 'unknown' row exists cleanly (remove existing then re-add)
unknown_row = spark.createDataFrame(
    [("unknown", "Unknown")],
    ["product_category_name", "product_category_name_english"]
)

trans_df = trans_df.filter(F.col("product_category_name") != "unknown").unionByName(unknown_row)
 
# 4.2.2 Trim both columns
for c in ("product_category_name", "product_category_name_english"):
    trans_df = trans_df.withColumn(c, F.trim(F.col(c)))
    trans_df = trans_df.withColumn(
        c, F.when(F.col(c) == "", None).otherwise(F.col(c))
    )
 
# 4.2.3 Remove rows with nulls in either of the key column
trans_df = trans_df.filter(
    F.col("product_category_name").isNotNull() &
    F.col("product_category_name_english").isNotNull()
)
 
# 4.2.4 Deduplicate on Portuguese name
trans_df = dedup(trans_df, "product_category_name")
 
# 4.2.5 Add the 2 known missing translations (idempotent: dedup handles re-runs to get apply latest filtering)
missing_trans = spark.createDataFrame(
    [
        ("portateis_cozinha_e_preparadores_de_alimentos", "Portable Kitchen Appliances"),
        ("pc_gamer", "PC Gamer"),
    ],
    ["product_category_name", "product_category_name_english"]
)
trans_df = dedup(trans_df.unionByName(missing_trans), "product_category_name")
 
write_silver(trans_df, "product_category_name_translation")

StatementMeta(, 8f8003aa-7acd-4e3c-8ee0-165d97e9724a, 21, Finished, Available, Finished, False)

bronze_products:     32,951 rows
bronze_translation:  71 rows
✅  silver_lakehouse.dbo.silver_products: 32,951 rows written
✅  silver_lakehouse.dbo.silver_product_category_name_translation: 74 rows written


In [20]:
## 5.0 Clean Geolocation table - Collapse/Combine Zip Codes

geo_df = spark.read.table("bronze_geolocation")
print(f"bronze_geolocation: {geo_df.count():,} rows (raw, with duplicates)")


# 5.1 One row per zip code - using avg coords, choosing MIN city/state
geo_df = geo_df.groupBy("geolocation_zip_code_prefix").agg(
    F.avg("geolocation_lat").alias("geolocation_lat"),
    F.avg("geolocation_lng").alias("geolocation_lng"),
    F.min("geolocation_city").alias("geolocation_city"),
    F.min("geolocation_state").alias("geolocation_state")
)
 
 # 5.2 Insert into silver table and verify
write_silver(geo_df, "geolocation")
print("   Expect ~19k rows after collapse")

StatementMeta(, 8f8003aa-7acd-4e3c-8ee0-165d97e9724a, 22, Finished, Available, Finished, False)

bronze_geolocation: 1,000,163 rows (raw, with duplicates)
✅  silver_lakehouse.dbo.silver_geolocation: 19,015 rows written
   Expect ~19k rows after collapse


In [21]:
## 6.0 Clean Customer Tables 

customers_df = spark.read.table("bronze_customers")
print(f"bronze_customers: {customers_df.count():,} rows")

# 6.1 Remove missing essential ID
customers_df = customers_df.filter(F.col("customer_id").isNotNull())

# 6.2 Trim columns
for c in ("customer_city", "customer_state"):
    customers_df = customers_df.withColumn(c, F.trim(F.col(c)))
    customers_df = customers_df.withColumn(
        c, F.when(F.col(c) == "", None).otherwise(F.col(c))
    )

# 6.3 Deduplicate rows on customer_id
customers_df = dedup(customers_df, "customer_id")
 
write_silver(customers_df, "customers")


StatementMeta(, 8f8003aa-7acd-4e3c-8ee0-165d97e9724a, 23, Finished, Available, Finished, False)

bronze_customers: 99,441 rows
✅  silver_lakehouse.dbo.silver_customers: 99,441 rows written


In [22]:
## 7.0 Clean Sellers table

sellers_df = spark.read.table("bronze_sellers")
print(f"bronze_sellers: {sellers_df.count():,} rows")
 
# 7.1 Remove rows missing essential ID
sellers_df = sellers_df.filter(F.col("seller_id").isNotNull())
 
# 7.2 Trim string columns
for c in ("seller_city", "seller_state"):
    sellers_df = sellers_df.withColumn(c, F.trim(F.col(c)))
    sellers_df = sellers_df.withColumn(
        c, F.when(F.col(c) == "", None).otherwise(F.col(c))
    )
 
# 7.3 Deduplicate rows on seller_id
sellers_df = dedup(sellers_df, "seller_id")
 
write_silver(sellers_df, "sellers")

StatementMeta(, 8f8003aa-7acd-4e3c-8ee0-165d97e9724a, 24, Finished, Available, Finished, False)

bronze_sellers: 3,095 rows
✅  silver_lakehouse.dbo.silver_sellers: 3,095 rows written


In [23]:
## 8.0  Validating Foreign Keys (after executing silver tables (1.0 - 7.0) successfully)

sv_orders   = spark.read.table("silver_orders")
sv_reviews  = spark.read.table("silver_order_reviews")
sv_products = spark.read.table("silver_products")
sv_trans    = spark.read.table("silver_product_category_name_translation")
 
# 8.1 Reviews → Orders
orphan_reviews = sv_reviews.join(sv_orders, "order_id", "left_anti").count()
print(f"🔍  Orphan reviews  (expect 0): {orphan_reviews}")
 
# 8.2 Products → Translation
orphan_cats = (sv_products
               .select("product_category_name").distinct()
               .join(sv_trans, "product_category_name", "left_anti")
               .count())
print(f"🔍  Untranslated categories (expect 0): {orphan_cats}")
 
print("\n🎉  FK validation complete.")

StatementMeta(, 8f8003aa-7acd-4e3c-8ee0-165d97e9724a, 25, Finished, Available, Finished, False)

🔍  Orphan reviews  (expect 0): 2701
🔍  Untranslated categories (expect 0): 0

🎉  FK validation complete.


In [24]:
## 9.0 Create analysis_df (joins of other tables)

sv_orders      = spark.read.table("silver_orders")
sv_order_items = spark.read.table("silver_order_items")
sv_products    = spark.read.table("silver_products")
sv_customers   = spark.read.table("silver_customers")
sv_sellers     = spark.read.table("silver_sellers")
 
# 9.1 Join 1: orders + order_items
analysis_df = sv_orders.join(sv_order_items, on="order_id", how="left")
 
# 9.2 Join 2: + products  —> use explicit column references to avoid ambiguity
analysis_df = analysis_df.join(
    sv_products.select(
        F.col("product_id"),
        F.col("product_category_name"),
        F.col("product_name_length"),
        F.col("product_description_length"),
        F.col("product_photos_qty"),
        F.col("product_weight_g"),
        F.col("product_length_cm"),
        F.col("product_height_cm"),
        F.col("product_width_cm"),
    ),
    on="product_id", #use common PK columns
    how="left"
)
 
# 9.3 Join 3: + customers
analysis_df = analysis_df.join(
    sv_customers.select(
        F.col("customer_id"),
        F.col("customer_unique_id"),
        F.col("customer_zip_code_prefix"),
        F.col("customer_city"),
        F.col("customer_state"),
    ),
    on="customer_id",
    how="left"
)
 
# 9.4 Join 4: + sellers
analysis_df = analysis_df.join(
    sv_sellers.select(
        F.col("seller_id"),
        F.col("seller_zip_code_prefix"),
        F.col("seller_city"),
        F.col("seller_state"),
    ),
    on="seller_id",
    how="left"
)


# 9.5 Output analysis_df and check if columns added correctly 
print(f"analysis_df row count: {analysis_df.count():,}")
print(f"Column count:          {len(analysis_df.columns)}")
print(f"Columns:               {analysis_df.columns}")
 
# 9.6 Sanity check: no duplicate column names
dupes = [c for c in analysis_df.columns if analysis_df.columns.count(c) > 1]
if dupes:
    print(f"⚠️  Duplicate columns found: {set(dupes)}")
else:
    print("✅  No duplicate columns.")

StatementMeta(, 8f8003aa-7acd-4e3c-8ee0-165d97e9724a, 26, Finished, Available, Finished, False)

analysis_df row count: 103,200
Column count:          29
Columns:               ['seller_id', 'customer_id', 'product_id', 'order_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'order_item_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name', 'product_name_length', 'product_description_length', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'seller_zip_code_prefix', 'seller_city', 'seller_state']
✅  No duplicate columns.


In [25]:
## 10.0 Creating columns for analysis_df (delay metrics / product vol/weight / freight band)

# 10.1 Reload if session was restarted (as a double confirm as pyspark cell execution is unstable)
if "analysis_df" not in dir():
    analysis_df = spark.read.table("silver_delivery_analysis")
    print("ℹ️  Reloaded analysis_df from silver_delivery_analysis")
 
# 10.2.1 Delivery delay
analysis_df = analysis_df.withColumn(
    "delay_days",
    F.datediff(
        F.col("order_delivered_customer_date"),
        F.col("order_estimated_delivery_date")
    )
)
 
# 10.2.2 Delivery status (only mark as Late if order was actually delivered)
analysis_df = analysis_df.withColumn(
    "delivery_status",
    F.when(F.col("order_delivered_customer_date").isNull(), "Not Delivered Yet")
     .when(F.col("delay_days") > 0, "Late")
     .when(F.col("delay_days") < 0, "Early")
     .otherwise("On Time")
)
 
# 10.2.3 Delay bucket (gonly for Late orders)
analysis_df = analysis_df.withColumn(
    "delay_bucket",
    F.when(F.col("delivery_status") != "Late", "Not Late")
     .when(F.col("delay_days") <= 3,  "1-3 Days")
     .when(F.col("delay_days") <= 7,  "4-7 Days")
     .when(F.col("delay_days") <= 14, "8-14 Days")
     .otherwise("15+ Days")
)
 
# 10.3.1 Product volume (cm³)
analysis_df = analysis_df.withColumn(
    "volume_cm3",
    F.col("product_length_cm") * F.col("product_height_cm") * F.col("product_width_cm")
)
 
# 10.3.2 Product weight (kg)
analysis_df = analysis_df.withColumn(
    "weight_kg",
    F.round(F.col("product_weight_g") / 1000, 3)
)
 
# 10.4.1 Freight band (delivery cost range)
analysis_df = analysis_df.withColumn(
    "freight_band",
    F.when(F.col("freight_value") < 15, "Low")
     .when(F.col("freight_value") < 30, "Medium")
     .otherwise("High")
)
 
# 10.4.2 Delivery route from Seller to Customer (e.g. "SP-RJ")
analysis_df = analysis_df.withColumn(
    "route",
    F.concat_ws("-", F.col("seller_state"), F.col("customer_state"))
)
 
# 10.5 Reference Check if above have been run successfully
print("Delivery status distribution:")
analysis_df.groupBy("delivery_status").count().orderBy("delivery_status").show()
 
print("Delay bucket distribution (Late only):")
(analysis_df
 .filter(F.col("delivery_status") == "Late")
 .groupBy("delay_bucket").count().orderBy("delay_bucket")
 .show())


StatementMeta(, 8f8003aa-7acd-4e3c-8ee0-165d97e9724a, 27, Finished, Available, Finished, False)

Delivery status distribution:
+-----------------+-----+
|  delivery_status|count|
+-----------------+-----+
|            Early|92199|
|             Late| 6666|
|Not Delivered Yet| 3005|
|          On Time| 1330|
+-----------------+-----+

Delay bucket distribution (Late only):
+------------+-----+
|delay_bucket|count|
+------------+-----+
|    1-3 Days| 1905|
|    15+ Days| 1408|
|    4-7 Days| 1852|
|   8-14 Days| 1501|
+------------+-----+



In [26]:
## 11.0 Create customer/seller longitude & latitude to count distance using Haversine formula

# 11.1 Reload if session was restarted (as a double confirm as pyspark cell execution is unstable)
if "analysis_df" not in dir():
    analysis_df = spark.read.table("silver_delivery_analysis")
    print("ℹ️  Reloaded analysis_df from silver_delivery_analysis")
 
sv_geo = spark.read.table("silver_geolocation")

# 11.2 Customer coordinates retrieval 
customer_coords = (
    spark.read.table("silver_customers")
    .select("customer_id", "customer_zip_code_prefix")
    .join(
        sv_geo.select(
            F.col("geolocation_zip_code_prefix").alias("customer_zip_code_prefix"),
            F.col("geolocation_lat").alias("customer_lat"),
            F.col("geolocation_lng").alias("customer_lng")
        ),
        on="customer_zip_code_prefix",
        how="left"
    )
    .drop("customer_zip_code_prefix") #no longer needed as customer lat lng is extracted
)
 
# 11.3 Seller coordinates retrieval
seller_coords = (
    spark.read.table("silver_sellers")
    .select("seller_id", "seller_zip_code_prefix")
    .join(
        sv_geo.select(
            F.col("geolocation_zip_code_prefix").alias("seller_zip_code_prefix"),
            F.col("geolocation_lat").alias("seller_lat"),
            F.col("geolocation_lng").alias("seller_lng")
        ),
        on="seller_zip_code_prefix",
        how="left"
    )
    .drop("seller_zip_code_prefix") #no longer needed as seller lat lng is extracted
)
 
# 11.4 Attach to analysis_df (drop existing coord cols first to prevent re-run duplicates)
coord_cols = {"customer_lat", "customer_lng", "seller_lat", "seller_lng",
              "distance_km", "distance_band"}

existing   = set(analysis_df.columns)

for c in coord_cols & existing:
    analysis_df = analysis_df.drop(c)
 
analysis_df = analysis_df.join(customer_coords, on="customer_id", how="left")
analysis_df = analysis_df.join(seller_coords,   on="seller_id",   how="left")
 
# 11.5 Creating Haversine User Define Function (UDF) for geolocation calculation
def haversine(lat1, lon1, lat2, lon2):
    if any(v is None for v in (lat1, lon1, lat2, lon2)):
        return None
    R = 6371.0
    phi1, phi2    = math.radians(lat1), math.radians(lat2)
    dphi          = math.radians(lat2 - lat1)
    dlambda       = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2)**2
    return round(2 * R * math.asin(math.sqrt(a)), 2)
 
haversine_udf = F.udf(haversine, "double")
 
analysis_df = analysis_df.withColumn(
    "distance_km",
    haversine_udf(
        F.col("seller_lat"), F.col("seller_lng"),
        F.col("customer_lat"), F.col("customer_lng")
    )
)
 
# 11.6 Distance band between seller & customer geolocation
analysis_df = analysis_df.withColumn(
    "distance_band",
    F.when(F.col("distance_km") < 50,   "0-50 km")
     .when(F.col("distance_km") < 100,  "50-100 km")
     .when(F.col("distance_km") < 500,  "100-500 km")
     .when(F.col("distance_km") < 1000, "500-1000 km")
     .otherwise("1000+ km")
)
 
print("✅  Coordinates and distance attached.")
print(f"   Columns: {analysis_df.columns}")

# 11.7 Validating if column results display correctly
analysis_df.select(
    F.col("order_id"),
    F.col("seller_state"),
    F.col("customer_state"),
    F.col("distance_km"),
    F.col("distance_band")
).show(20, truncate = False)


StatementMeta(, 8f8003aa-7acd-4e3c-8ee0-165d97e9724a, 28, Finished, Available, Finished, False)

✅  Coordinates and distance attached.
   Columns: ['seller_id', 'customer_id', 'product_id', 'order_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'order_item_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name', 'product_name_length', 'product_description_length', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'delay_days', 'delivery_status', 'delay_bucket', 'volume_cm3', 'weight_kg', 'freight_band', 'route', 'customer_lat', 'customer_lng', 'seller_lat', 'seller_lng', 'distance_km', 'distance_band']
+--------------------------------+------------+--------------+-----------+-------------+
|order_id                        |seller_state|customer_stat

In [27]:
## 12.0 Validation of silver_delivery_analysis

if "analysis_df" not in dir():
    raise ValueError("analysis_df not found — run Cells 10-11 first.")

# 12.1 Save previously done work as delivery_analysis
write_silver(analysis_df, "delivery_analysis")
 
# 12.2 Reload from saved table to validate persisted data
saved = spark.read.table("silver_delivery_analysis")
 
print(f"\nFinal row count: {saved.count():,}")
print(f"Final columns ({len(saved.columns)}): {saved.columns}")
 
# 12.3 Validate Distance summary
print("\n--- Distance column summary ---")
saved.select(
    F.count("distance_km").alias("non_null_count"),
    F.sum(F.col("distance_km").isNull().cast("int")).alias("null_count"),
    F.round(F.min("distance_km"), 2).alias("min_km"),
    F.round(F.avg("distance_km"), 2).alias("avg_km"),
    F.round(F.max("distance_km"), 2).alias("max_km")
).show()
 
# 12.4 Validate Late rate by distance band (delivered orders only)
print("--- Late rate by distance band ---")
(saved
 .filter(F.col("order_delivered_customer_date").isNotNull())
 .groupBy("distance_band")
 .agg(
     F.count("*").alias("total_orders"),
     F.sum(F.when(F.col("delivery_status") == "Late", 1).otherwise(0)).alias("late_orders")
 )
 .withColumn("late_rate_percentage (%)", F.round(F.col("late_orders") / F.col("total_orders") * 100, 2))
 .orderBy("distance_band")
 .show()
)
 
# 12.5 Validate Delivery status distribution
print("--- Delivery status distribution ---")
saved.groupBy("delivery_status").count().orderBy("delivery_status").show()
 
print("\n🎉  silver_delivery_analysis is complete and validated.")

StatementMeta(, 8f8003aa-7acd-4e3c-8ee0-165d97e9724a, 29, Finished, Available, Finished, False)

✅  silver_lakehouse.dbo.silver_delivery_analysis: 103,200 rows written

Final row count: 103,200
Final columns (48): ['seller_id', 'customer_id', 'product_id', 'order_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'order_item_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name', 'product_name_length', 'product_description_length', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'delay_days', 'delivery_status', 'delay_bucket', 'volume_cm3', 'weight_kg', 'freight_band', 'route', 'customer_lat', 'customer_lng', 'seller_lat', 'seller_lng', 'distance_km', 'distance_band', 'is_interstate', 'purchase_year', 'purchase_month', 'purchase_quarter', 'purchase_

In [28]:
## 13.0 Creating Interstate flag (delivers across state / inside same state)
from pyspark.sql import functions as F

# 13.0.1 Safe reload of analysis_df incase fabric disconnects
analysis_df = spark.read.table("silver_lakehouse.dbo.silver_delivery_analysis")
print(f"Loaded silver_delivery_analysis: {analysis_df.count():,} rows")
print(f"Columns before enrichment ({len(analysis_df.columns)}): {analysis_df.columns}")

# 13.0.2 Safe rerun without duplicates

cell14_cols = [
    "is_interstate", "purchase_year", "purchase_month",
    "purchase_quarter", "purchase_day_of_week", "is_weekend_purchase"
]
for c in cell14_cols:
    if c in analysis_df.columns:
        analysis_df = analysis_df.drop(c)
 
print(f"\nColumns after dropping old Cell 14 cols: {len(analysis_df.columns)}")


# 13.1 Creating Cross-state deliveries column (key hypothesis for analysis)
analysis_df = analysis_df.withColumn(
    "is_interstate",
    F.when(F.col("seller_state") != F.col("customer_state"), True).otherwise(False)
)
 
# 13.2 Validation if states output between interstate vs same-state
print("\n--- Interstate vs same-state orders ---")
analysis_df.groupBy("is_interstate").count().orderBy("is_interstate").show()
 
# 13.3 Validate late delivery that has been delivered between interstate vs same-state
print("--- Late rate: interstate vs same-state (delivered orders only) ---")
(analysis_df
 .filter(F.col("order_delivered_customer_date").isNotNull())
 .groupBy("is_interstate")
 .agg(
     F.count("*").alias("total_orders"),
     F.sum(F.when(F.col("delivery_status") == "Late", 1).otherwise(0)).alias("late_orders")
 )
 .withColumn("late_rate_percentage (%)", F.round(F.col("late_orders") / F.col("total_orders") * 100, 2))
 .orderBy("is_interstate")
 .show()
)
 
 
# 13.4 Time-based columns for seasonal analysis

# 13.4.1 Purchase year
analysis_df = analysis_df.withColumn(
    "purchase_year",
    F.year(F.col("order_purchase_timestamp"))
)

# 13.4.2 Purchase month
analysis_df = analysis_df.withColumn(
    "purchase_month",
    F.month(F.col("order_purchase_timestamp"))
)

# 13.4.3 Purchase quater
analysis_df = analysis_df.withColumn(
    "purchase_quarter",
    F.quarter(F.col("order_purchase_timestamp"))
)

# 13.4.4 Purchase day of the week
analysis_df = analysis_df.withColumn(
    "purchase_day_of_week",
    F.dayofweek(F.col("order_purchase_timestamp"))   # 1=Sun, 7=Sat
)

# 13.4.5 Purchase during weekend
analysis_df = analysis_df.withColumn(
    "is_weekend_purchase",
    F.when(F.col("purchase_day_of_week").isin(1, 7), True).otherwise(False)
)
 


# 13.5 Time-based columns for seasonal analysis

# 13.5.1 Validate: late rate by month
print("--- Late rate by purchase month ---")
(analysis_df
 .filter(F.col("order_delivered_customer_date").isNotNull())
 .groupBy("purchase_month")
 .agg(
     F.count("*").alias("total_orders"),
     F.sum(F.when(F.col("delivery_status") == "Late", 1).otherwise(0)).alias("late_orders")
 )
 .withColumn("late_rate_percentage (%)", F.round(F.col("late_orders") / F.col("total_orders") * 100, 2))
 .orderBy("purchase_month")
 .show()
)
 
# 13.5.2 Validate: late rate by quarter
print("--- Late rate by quarter ---")
(analysis_df
 .filter(F.col("order_delivered_customer_date").isNotNull())
 .groupBy("purchase_quarter")
 .agg(
     F.count("*").alias("total_orders"),
     F.sum(F.when(F.col("delivery_status") == "Late", 1).otherwise(0)).alias("late_orders")
 )
 .withColumn("late_rate_pct", F.round(F.col("late_orders") / F.col("total_orders") * 100, 2))
 .orderBy("purchase_quarter")
 .show()
)
 
 
# 13.6 Overwrite silver_delivery_analysis with all new columns
(analysis_df
 .write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("silver_lakehouse.dbo.silver_delivery_analysis"))

# 13.7.1 Validation for missing columns

saved = spark.read.table("silver_lakehouse.dbo.silver_delivery_analysis")
print(f"\nVerifying saved table...")
print(f"Row count: {saved.count():,}")
 
for c in cell14_cols:
    status = "✅" if c in saved.columns else "❌  MISSING"
    print(f"  {status}  {c}")

# 13.7.2 Validation Overview
final_count = spark.read.table("silver_delivery_analysis").count()
print(f"\n✅  silver_delivery_analysis updated: {final_count:,} rows")
print(f"    Final columns ({len(analysis_df.columns)}): {analysis_df.columns}")
print("\n🎉  Silver layer 100% complete. Ready for Gold layer.")


StatementMeta(, 8f8003aa-7acd-4e3c-8ee0-165d97e9724a, 30, Finished, Available, Finished, False)

Loaded silver_delivery_analysis: 103,200 rows
Columns before enrichment (42): ['seller_id', 'customer_id', 'product_id', 'order_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'order_item_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name', 'product_name_length', 'product_description_length', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'delay_days', 'delivery_status', 'delay_bucket', 'volume_cm3', 'weight_kg', 'freight_band', 'route', 'customer_lat', 'customer_lng', 'seller_lat', 'seller_lng', 'distance_km', 'distance_band']

Columns after dropping old Cell 14 cols: 42

--- Interstate vs same-state orders ---
+-------------+-----+
|is_interstat